# Notes:

This notebook aims to calculate the association between structural change and entanglement.
We also control for length as the confounding factor.

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as ss
from statsmodels.stats.contingency_tables import Table2x2
import statsmodels.formula.api as smf

In [2]:
def fmt_p(p):
    """Mixed formatting for p-values."""
    return f"{p:.3e}" if p < 0.001 else f"{p:.3f}"

def summarize_logit(result, alpha=0.05, exponentiate=True):
    """
    Summarize statsmodels Logit/GLM(Binomial) results.

    Returns a DataFrame with:
    coef, OR, CI, numeric p-value, and formatted p-value.
    """
    params = result.params
    conf = result.conf_int(alpha=alpha)   # columns: [lower, upper]
    pvals = result.pvalues

    df = pd.DataFrame({
        "coef": params,
        "ci_lower": conf.iloc[:, 0],
        "ci_upper": conf.iloc[:, 1],
        "pvalue": pvals
    })

    if exponentiate:
        df["OR"] = np.exp(df["coef"])
        df["OR_ci_lower"] = np.exp(df["ci_lower"])
        df["OR_ci_upper"] = np.exp(df["ci_upper"])

    # add formatted p-value column
    df["p_fmt"] = df["pvalue"].apply(fmt_p)

    # nicer column order
    if exponentiate:
        df = df[[
            "coef",
            "OR",
            "OR_ci_lower",
            "OR_ci_upper",
            "pvalue",
            "p_fmt"
        ]]
    else:
        df = df[["coef", "ci_lower", "ci_upper", "pvalue", "p_fmt"]]

    return df

## Load data

In [3]:
df_SC_Ent= pd.read_pickle('../data/SC_Ent.pkl')

In [4]:
df_SC_Ent

,SGDID,Uniprot,SC,length_AF,entangled,asphericity,fraction_idp,Knot,cov_lasso,entangled_residues,Cutsite_residues,clustered_entangled_residues
0,S000002468,Q12298,0,539,1,0.139968,0.011132,0,0,"[26, 27, 28, 31, 33, 34, 35, 36, 37, 38, 39, 4...",[],"[27, 31, 34, 35, 36, 38, 39, 40, 41, 42, 43, 4..."
1,S000000094,P35842,0,467,1,0.097823,0.051392,0,0,"[21, 23, 30, 31, 32, 33, 34, 35, 37, 38, 41, 4...",[],"[32, 33, 34, 35, 37, 38, 41, 46, 47, 67, 68, 6..."
2,S000002490,P38961,0,392,1,0.108081,0.295918,0,0,"[177, 178, 179, 180, 181, 182, 183, 184, 190, ...",[],"[181, 193, 266, 267, 268, 290, 292, 293, 294, ..."
3,S000003653,P46956,0,311,1,0.311604,0.099678,0,0,"[64, 65, 68, 69, 70, 71, 72, 73, 74, 75, 78, 1...",[],"[203, 210, 211, 212, 213, 252, 253, 254, 255, ..."
4,S000002198,P12945,0,854,0,0.074009,0.069087,0,0,[],[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...
2251,S000001943,P43619,0,295,1,0.383145,0.071186,0,0,"[1, 2, 3, 4, 5, 8, 10, 12, 13, 14, 15, 16, 17,...",[],"[15, 18, 19, 20, 21, 22, 23, 24, 25, 26, 48, 4..."
2252,S000003720,P46984,0,123,0,0.230204,1.000000,0,0,[],[],[]
2253,S000005560,Q12013,0,749,0,0.224206,0.157543,0,0,[],[],[]
2254,S000005185,P11412,1,505,1,0.142002,0.017822,0,0,"[6, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, ...","[394, 395, 396, 397, 398, 399, 400, 401, 402]","[10, 11, 12, 13, 14, 15, 16, 17, 18, 23, 24, 2..."


# Get OR using contingency table

In [5]:
SC_Ent = len(df_SC_Ent[(df_SC_Ent.SC==1) & (df_SC_Ent.entangled == 1)])
NSC_NEnt = len(df_SC_Ent[(df_SC_Ent.SC==0) & (df_SC_Ent.entangled == 0)])
SC_NEnt = len(df_SC_Ent[(df_SC_Ent.SC==1) & (df_SC_Ent.entangled == 0)])
NSC_Ent = len(df_SC_Ent[(df_SC_Ent.SC==0) & (df_SC_Ent.entangled == 1)])

print("Contingency table (rows: entanglement, columns: SC)")
print(f"{'':<15} {'SC':>10} {'Non-SC':>10}")
print(f"{'Entangled':<15} {SC_Ent:>10} {NSC_Ent:>10}")
print(f"{'Non-entangled':<15} {SC_NEnt:>10} {NSC_NEnt:>10}")

print('-'*50)
table = [[SC_Ent, NSC_Ent],
        [SC_NEnt, NSC_NEnt]]

ct = Table2x2(table)

odds_ratio, pvalue = ss.fisher_exact(table)
ci_low, ci_high = ct.oddsratio_confint(alpha=0.05, method="exact")

print(f"Odds Ratio: {odds_ratio:.3f}")
print(f"95% CI: [{ci_low:.3f}, {ci_high:.3f}]")
print(f"P-value: {pvalue}")

Contingency table (rows: entanglement, columns: SC)
                        SC     Non-SC
Entangled              368       1318
Non-entangled           70        500
--------------------------------------------------
Odds Ratio: 1.994
95% CI: [1.514, 2.627]
P-value: 3.252407554089501e-07


In [6]:
print("The prevalence of age-associated structural changes across the set of mass-spectrometric-observed proteins:")
print(f"Structural change proteins: {SC_Ent + SC_NEnt}")
print(f"Total Proteins: {SC_Ent + NSC_Ent + SC_NEnt + NSC_NEnt}")
print(f"Percentage: {100*(SC_Ent + SC_NEnt)/(SC_Ent + NSC_Ent + SC_NEnt + NSC_NEnt):.3f} %")

The prevalence of age-associated structural changes across the set of mass-spectrometric-observed proteins:
Structural change proteins: 438
Total Proteins: 2256
Percentage: 19.415 %


In [7]:
print("Prevalent of age-associated structural change protein in entangled proteins:")
print(f"Entangled proteins exhibit structural change: {SC_Ent}")
print(f"Total Entangled Proteins: {(SC_Ent+NSC_Ent)}")
print(f"Percentage: {100*(SC_Ent)/(SC_Ent+NSC_Ent):.3f} %")

Prevalent of age-associated structural change protein in entangled proteins:
Entangled proteins exhibit structural change: 368
Total Entangled Proteins: 1686
Percentage: 21.827 %


In [8]:
print("Prevalent of age-associated structural change protein in non-entangled proteins:")
print(f"Non-entangled proteins exhibit Structural change: {SC_NEnt}")
print(f"Total Non-entangled Proteins: {(SC_NEnt+NSC_NEnt)}")
print(f"Percentage: {100*(SC_NEnt)/(SC_NEnt+NSC_NEnt):.3f} %")

Prevalent of age-associated structural change protein in non-entangled proteins:
Non-entangled proteins exhibit Structural change: 70
Total Non-entangled Proteins: 570
Percentage: 12.281 %


# Logistic Regression

In [9]:
# Standardize 'length_AF' using z-score
df_SC_Ent['length_z'] = (df_SC_Ent['length_AF'] - df_SC_Ent['length_AF'].mean()) / df_SC_Ent['length_AF'].std()

In [10]:
df_SC_Ent

,SGDID,Uniprot,SC,length_AF,entangled,asphericity,fraction_idp,Knot,cov_lasso,entangled_residues,Cutsite_residues,clustered_entangled_residues,length_z
0,S000002468,Q12298,0,539,1,0.139968,0.011132,0,0,"[26, 27, 28, 31, 33, 34, 35, 36, 37, 38, 39, 4...",[],"[27, 31, 34, 35, 36, 38, 39, 40, 41, 42, 43, 4...",0.179299
1,S000000094,P35842,0,467,1,0.097823,0.051392,0,0,"[21, 23, 30, 31, 32, 33, 34, 35, 37, 38, 41, 4...",[],"[32, 33, 34, 35, 37, 38, 41, 46, 47, 67, 68, 6...",-0.024684
2,S000002490,P38961,0,392,1,0.108081,0.295918,0,0,"[177, 178, 179, 180, 181, 182, 183, 184, 190, ...",[],"[181, 193, 266, 267, 268, 290, 292, 293, 294, ...",-0.237167
3,S000003653,P46956,0,311,1,0.311604,0.099678,0,0,"[64, 65, 68, 69, 70, 71, 72, 73, 74, 75, 78, 1...",[],"[203, 210, 211, 212, 213, 252, 253, 254, 255, ...",-0.466648
4,S000002198,P12945,0,854,0,0.074009,0.069087,0,0,[],[],[],1.071726
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2251,S000001943,P43619,0,295,1,0.383145,0.071186,0,0,"[1, 2, 3, 4, 5, 8, 10, 12, 13, 14, 15, 16, 17,...",[],"[15, 18, 19, 20, 21, 22, 23, 24, 25, 26, 48, 4...",-0.511978
2252,S000003720,P46984,0,123,0,0.230204,1.000000,0,0,[],[],[],-0.999271
2253,S000005560,Q12013,0,749,0,0.224206,0.157543,0,0,[],[],[],0.774250
2254,S000005185,P11412,1,505,1,0.142002,0.017822,0,0,"[6, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, ...","[394, 395, 396, 397, 398, 399, 400, 401, 402]","[10, 11, 12, 13, 14, 15, 16, 17, 18, 23, 24, 2...",0.082974


In [11]:
model = smf.logit('SC ~ entangled + length_z', data=df_SC_Ent)
result = model.fit()
print(result.summary())

Optimization terminated successfully.
         Current function value: 0.484882
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                     SC   No. Observations:                 2256
Model:                          Logit   Df Residuals:                     2253
Method:                           MLE   Df Model:                            2
Date:                Mon, 02 Mar 2026   Pseudo R-squ.:                 0.01483
Time:                        15:20:23   Log-Likelihood:                -1093.9
converged:                       True   LL-Null:                       -1110.4
Covariance Type:            nonrobust   LLR p-value:                 7.025e-08
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -2.0467      0.132    -15.452      0.000      -2.306      -1.787
entangled      0.7907      0.

In [12]:
summary_df = summarize_logit(result)
print(summary_df.to_string(float_format=lambda x: f"{x:.3f}"))

            coef    OR  OR_ci_lower  OR_ci_upper  pvalue      p_fmt
Intercept -2.047 0.129        0.100        0.167   0.000  7.357e-54
entangled  0.791 2.205        1.654        2.938   0.000  6.818e-08
length_z  -0.146 0.864        0.767        0.974   0.017      0.017


# calculate the odds

In [13]:
# Odd of entangled proteins exhibit age-associated structural changes
# Coefficients and covariance matrix
params = result.params
cov = result.cov_params()

# Example: calculate odds and CI for entangled = 1, length_z = 0
a = np.array([1, 1, 0])  # intercept, entangled, length_z

# Compute logit
logit = np.dot(a, params)

# Variance and standard error
var_logit = np.dot(a, np.dot(cov, a))
se_logit = np.sqrt(var_logit)

# CI in logit scale
logit_lower = logit - 1.96 * se_logit
logit_upper = logit + 1.96 * se_logit

# Convert to odds
odds = np.exp(logit)
ci_lower = np.exp(logit_lower)
ci_upper = np.exp(logit_upper)

print(f"Odds: {odds:.5f} (95% CI: [{ci_lower:.5f}, {ci_upper:.5f}]")

Odds: 0.28478 (95% CI: [0.25352, 0.31990]


In [14]:
# Odd of non-entangled proteins exhibit age-associated structural change
# Coefficients and covariance matrix
params = result.params
cov = result.cov_params()

# Example: calculate odds and CI for entangled = 0, length_z = 0
a = np.array([1, 0, 0])  # intercept, entangled, length_z

# Compute logit
logit = np.dot(a, params)

# Variance and standard error
var_logit = np.dot(a, np.dot(cov, a))
se_logit = np.sqrt(var_logit)

# CI in logit scale
logit_lower = logit - 1.96 * se_logit
logit_upper = logit + 1.96 * se_logit

# Convert to odds
odds = np.exp(logit)
ci_lower = np.exp(logit_lower)
ci_upper = np.exp(logit_upper)

print(f"Odds: {odds:.5f} (95% CI: [{ci_lower:.5f}, {ci_upper:.5f}]")

Odds: 0.12916 (95% CI: [0.09963, 0.16745]
